# PyRIT Initializers

You can configure PyRIT using:
1. **Built-in initializers** - TargetInitializer, ScorerInitializer, TechniqueInitializer, LoadDefaultDatasets
2. **External scripts** - Custom PyRITInitializer classes for project-specific needs

## Execution Order

When `initialize_pyrit_async` is called:
1. Environment files are loaded (`.env`, `.env.local`)
2. Memory database is configured
3. Initializers execute in the order they are passed

## Creating an Initializer

The following is a minimal `PyRITInitializer` class. It doesn't need much! In this case, it sets the default value for temperature for all OpenAIChatTargets to .9.

In [ ]:
from pyrit.common.apply_defaults import set_default_value
from pyrit.prompt_target import OpenAIChatTarget
from pyrit.setup.pyrit_initializer import PyRITInitializer


class CustomInitializer(PyRITInitializer):
    """Sets custom temperature for OpenAI targets."""

    async def initialize_async(self) -> None:
        set_default_value(class_type=OpenAIChatTarget, parameter_name="temperature", value=0.9)


CustomInitializer()

## Declaring Parameters

Declare inputs with a `supported_parameters()` classmethod. The same `Parameter`
model and resolver used by scenarios convert values, validate declarations,
reject unknown names, and supply defaults. `self.params` contains typed values,
not lists of strings for every parameter.

CLI syntax is unchanged: `target:tags=default,scorer;auto_group=false` supplies
a list of tags and a boolean. YAML can supply native scalars and lists.
An omitted or null value uses its declared default.

For a required input, use `default=REQUIRED_VALUE` from `pyrit.common`.
Construction leaves required values as `None` so catalog metadata remains
available. `set_params_from_args` rejects missing or null required inputs.
Optional defaults are available immediately, including on directly constructed
initializers.

Configuration converts inputs once. `validate_params()` and setup validation
check the resolved bag without changing it. Use `set_params_from_args` instead
of assigning a raw or partial `params` dictionary.

Integer inputs reject fractional numbers instead of truncating them. For
example, `RefreshDatasets` rejects `days: 0.5`; it cannot become the
unconditional-refresh setting `days: 0`.

**Migration for custom initializers:** replace the old `@property` declaration
with `@classmethod` and use `cls` for class constants. Declare `param_type` and
read scalar values directly, such as `self.params["temperature"]`, not `[0]`.
Use `set_params_from_args` to configure values before initialization.

In [ ]:
from pyrit.models import Parameter


class ParameterizedInitializer(PyRITInitializer):
    """Configures the default OpenAI temperature."""

    @classmethod
    def supported_parameters(cls) -> list[Parameter]:
        return [Parameter(name="temperature", description="Sampling temperature", param_type=float, default=0.9)]

    async def initialize_async(self) -> None:
        set_default_value(
            class_type=OpenAIChatTarget,
            parameter_name="temperature",
            value=self.params["temperature"],
        )


configured_initializer = ParameterizedInitializer()
configured_initializer.set_params_from_args(args={"temperature": "0.7"})
assert configured_initializer.params["temperature"] == 0.7

## Built-in Initializers

PyRIT includes a few built-in initializers that set more intelligent defaults!

- **TargetInitializer**: Registers targets from environment variables. With only OPENAI_CHAT_ENDPOINT, OPENAI_CHAT_MODEL, and OPENAI_CHAT_KEY set, it registers a sensible default objective/converter target.
- **ScorerInitializer**: Registers default scorers (run it after TargetInitializer, since scorers use those targets).
- **TechniqueInitializer**: Registers the attack techniques used by scenarios.
- **LoadDefaultDatasets**: Loads the datasets required by registered scenarios into memory.

These are easy to include.

In [ ]:
from pyrit.setup import initialize_pyrit_async
from pyrit.setup.initializers import ScorerInitializer, TargetInitializer

# Using built-in initializers
await initialize_pyrit_async(  # type: ignore
    memory_db_type="InMemory", initializers=[TargetInitializer(), ScorerInitializer()]
)

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


No new upgrade operations detected.


## External Scripts

External scripts allow custom configurations without modifying PyRIT. For example, you can write your own library, include them, and never have to check out pyrit in editable mode. Here are some use cases:
- Custom targets for security assessments
- Project-specific defaults
- Organization-specific defaults

As an example, say you are building a product, and want to set all your `adversarial_chat` in one place. You can using this!

Like the built-in initializers, external scripts have the same format and must contain PyRITInitializer classes. In fact, using something like TargetInitializer() as a template for your own is not a bad place to start.

In [ ]:
import os
import shutil
import tempfile

from pyrit.setup import initialize_pyrit_async

temp_dir = tempfile.mkdtemp()
script_path = os.path.join(temp_dir, "custom_init.py")

# This is the simple custom initializer from the "Creating an Initializer" section of this notebook
script_content = """
from pyrit.setup.pyrit_initializer import PyRITInitializer
from pyrit.common.apply_defaults import set_default_value
from pyrit.prompt_target import OpenAIChatTarget

class CustomInitializer(PyRITInitializer):
    \"\"\"Sets custom temperature for OpenAI targets.\"\"\"

    async def initialize_async(self) -> None:
        set_default_value(class_type=OpenAIChatTarget, parameter_name="temperature", value=0.9)

"""

with open(script_path, "w") as f:
    f.write(script_content)

print(f"Created: {script_path}")

await initialize_pyrit_async(  # type: ignore
    memory_db_type="InMemory", initialization_scripts=[temp_dir + "/custom_init.py"]
)

if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)

Created: ./AppData/Local/Temp/tmprs016r1e/custom_init.py
Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


The initialization_scripts argument ultimately uses `pathlib.Path`, so the scripts are loaded relative to the current working directory (where you're executing the script from, not where PyRIT library is). To avoid ambiguity, it is usually better to use full paths if possible.

## More information:
- [Configuration notebook](1_configuration.ipynb) shows practical examples with custom targets
- [Default Values notebook](default_values.md) explains how defaults work
